<!--
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
-->

# GÜN 064: Class Templates (Sınıf Şablonları, CTAD Çıkarsama Kılavuzları, Kısmi Özelleştirme, İki Aşamalı İsim Çözümleme ve Sıfır Dinamik Tahsisli Aviyonik Konteynerler)

[![License: All Rights Reserved](https://img.shields.io/badge/license-All%20Rights%20Reserved-red?style=flat-square)](https://github.com/seydivakkas)
[![C++ Standard](https://img.shields.io/badge/C%2B%2B-20-blue?style=flat-square&logo=c%2B%2B)](https://en.cppreference.com/w/cpp/20)
[![Domain](https://img.shields.io/badge/Domain-Avionics%20Static%20Containers%20%26%20Guidance%20Matrices-darkgreen?style=flat-square)]()
[![Safety](https://img.shields.io/badge/Safety-MISRA%20C%2B%2B%3A2023%20%7C%20DO--178C%20DAL--A-orange?style=flat-square)]()

---

## 🏛️ Akademik ve Kurumsal Bildirim
Bu modül, **ASELSAN, HAVELSAN, ROKETSAN, TUSAŞ ve STM** gibi savunma sanayii devlerinin görev-kritik aviyonik uçuş kontrol, radar hedef izleme ve füze güdüm matris hesaplama mimarilerinde sıfır dinamik bellek tahsisi (*zero heap allocation*) ile tip güvenli jenerik konteynerler ve matematiksel yapılar inşa etmek üzere tasarlanmıştır.

- **Müfredat Fazı:** Faz 1 (F1) — Modern C++20 Çekirdek Sistem Programlama
- **Ön Koşul Konu:** Gün 063 (`Function templates` ve tip çıkarsama kuralları)
- **Pedagojik Derinlik:** 40-Nokta Tam Kapsam Doğrulaması (First Principles -> Kısmi Özelleştirme -> Two-Phase Lookup -> Sabotaj -> Benchmarking -> Sektörel Standartlar)


## 1. 🎯 Bugünkü Problem: Konteynerlerde Tip Ayrımı, Dinamik Bellek İsrafı ve Fabrika Fonksiyonu Zorunluluğu

Sistem mimarisinde bir dairesel tampon (`RingBuffer`), sabit boyutlu dizi (`StaticArray`) veya matematiksel dönüşüm matrisi (`Matrix`) tasarlarken üç temel sorunla karşılaşırız:

### 1. `void*` Tabanlı C Konteynerleri ve Tip Körlüğü
Geleneksel C dilinde jenerik bir kuyruk veya yığın yazmak için veriler `void*` olarak saklanırdı:
```c
typedef struct { void** data; size_t capacity; } GenericQueue;
```
- **Felaket:** Kuyruğa `RadarTrack*` atıp yanlışlıkla `WeaponStatus*` olarak çekildiğinde derleyici hiçbir uyarı vermez. Bellek bozulur, füzeler yanlış hedefe ateşlenir. Ayrıca her eleman için heap üzerinden `malloc` çağrısı zorunlu hale gelir.

### 2. C++17 Öncesi Şablon Başlatma Hantallığı
C++11/14'te sınıf şablonlarının tür parametrelerini her seferinde elle yazmak zorunluydu:
```cpp
std::pair<int, double> p(1, 3.14); // Uzun ve gereksiz tekrar!
```
Yazılımcılar bu hantallıktan kaçmak için `std::make_pair`, `std::make_tuple` gibi yardımcı fabrika fonksiyonları türetmek zorunda kalırdı.

### 3. Kısmi Özelleştirme (Partial Specialization) Eksikliği
Fonksiyon şablonları kısmi olarak özelleştirilemez (*partial specialization is forbidden for function templates*). Örneğin bir algoritmayı yalnızca işaretçiler (`T*`) veya sabit diziler (`T[N]`) için optimize etmek istediğimizde fonksiyon şablonları yetersiz kalır.

---

### Bugünkü Çözüm: C++20 Class Templates (Sınıf Şablonları)
- **Kapsamlı Parametrik Veri Yapıları:** `template<typename T, size_t N> class StaticVector;` ile tür ve boyut derleme zamanında mühürlenir.
- **CTAD (Class Template Argument Deduction):** C++17/20 ile artık `std::pair p(1, 3.14);` yazmak yeterlidir; kurucudan tür otomatik çıkarılır.
- **Güçlü Kısmi Özelleştirme:** Sınıf şablonları işaretçiler (`T*`), referanslar veya özel türler için derleme anında kısmen özelleştirilebilir (*Zero-overhead custom memory layouts*).


## 2. ❓ Bu Konuyu Neden Öğreniyoruz? (Savunma Sanayii Gerekçesi)

Savunma ve aviyonik yazılımlarında sınıf şablonları iki hayati sütuna dayanır:

### 1. DO-178C DAL-A Uyumlu Sabit Boyutlu Konteynerler
Aviyonik uçuş kontrol bilgisayarlarında (FCS) `std::vector` kullanımı çalışma zamanında heap tahsisi (`malloc`) yaptığı ve belirsiz gecikme dalgalanması (*jitter*) yarattığı için **kesinlikle yasaktır**.
- Bunun yerine `StaticVector<FlightCommand, 64>` sınıf şablonu kullanılır.
- Kapasite ve tür şablon parametresi olarak verilir. Bellek tamamen nesnenin kendi gövdesinde (stack veya data segmentinde) yaşar.
- Sıfır heap tahsisi, sıfır bellek parçalanması ve %100 deterministik yürütme garanti edilir.

### 2. Füze Güdümünde Yüksek Hızlı Matris ve Kuaterniyon Hesaplamaları
ROKETSAN güdüm algoritmalarında koordinat dönüşümleri $3 	imes 3$ yönelim matrisleri (`Matrix<double, 3, 3>`) ve $4 	imes 1$ kuaterniyon vektörleri ile yapılır.
- Sınıf şablonu boyutları derleme zamanında bildiği için döngüleri tamamen açar (*loop unrolling*).
- $3 	imes 3$ matris çarpımı 9 adet doğrudan SIMD FMA (Fused Multiply-Add) assembly komutuna indirgenir; çalışma zamanı matris indirection maliyeti sıfıra iner!


## 3. 🔍 5N1K Analizi (Sistem ve Donanım Sınırları)

| Soru | Sistem & Donanım Mühendisliği Yanıtı |
| :--- | :--- |
| **Ne? (What)** | **Class Template (Sınıf Şablonu)**, üye değişkenleri ve fonksiyonları derleme zamanı tür ve sabit parametrelerine bağlı olan, çağrıldığı her parametre kümesi için bağımsız ve optimize edilmiş bir sınıf türü üreten C++ metaprogramlama yapısıdır. |
| **Neden? (Why)** | Tip güvenli jenerik konteynerler üretmek, dinamik bellek tahsisini ortadan kaldırarak stack üzerinde sabit boyutlu yapılar kurmak ve kısmi özelleştirme ile donanıma en yakın bellek düzenini sağlamak için. |
| **Nasıl? (How)** | `template<typename T, size_t Size>` sözdizimi ile tanımlanır. C++17/20 CTAD kuralları ile kurucu argümanlarından tür çıkarılır veya açıkça `Container<int, 10>` şeklinde somutlaştırılır. |
| **Nerede? (Where)** | TUSAŞ uçuş kontrol sabit tamponlarında, ASELSAN radar hedef korelasyon tablolarında, ROKETSAN güdüm matrislerinde ve Eigen lineer cebir kütüphanesinde. |
| **Ne Zaman? (When)** | Bir veri yapısının veya sınıfın mantığı içerdiği türden ve boyuttan bağımsız olduğunda ve sıfır çalışma zamanı ek yükü istendiğinde. |
| **Kim? (Who)** | Aviyonik veri yapısı mimarları, DSP algoritma mühendisleri, gömülü sistem programcıları ve DO-178C sertifikasyon uzmanları. |


## 4. 💡 Günlük Hayat Benzetmesi (Zihinsel Model / Analoji)

> ### 📦 Modüler Fabrika Saklama Sandıkları ve Boyut Kalıpları Analojisi
> 
> Bir askeri lojistik ikmal deposunu düşünün:
> 
> 1. **Eski C / void* Yolu (Kör Çuvallar):**
>    Depoda yalnızca üzeri yazısız büyük bez çuvallar vardır (`void*`). Asker içine tüfek mermisi de koyar, hassas telsiz devresi de, el bombası da. Çuvalın içinde ne olduğu dışarıdan bilinmez. Bir asker mermi ararken elini daldırıp el bombasının pimini çekebilir (**Tip Körlüğü ve Çöküş**).
> 
> 2. **Ayrı Ayrı Sınıflar Yolu (100 Farklı Marangoz Siparişi):**
>    Depocu 5.56mm mermi için ayrı bir tahta sandık, 7.62mm mermi için ayrı bir sandık, 120mm top mermisi için üçüncü bir sandık yaptırır. Her yeni mühimmat çıktığında marangoz sıfırdan aynı sandık tasarımını kopyalar (**Kod Tekrarı ve Bakım Kabusu**).
> 
> 3. **Modern Class Template Yolu (Endüstriyel Modüler Pres Kalıbı):**
>    Fabrikada tek bir akıllı çelik pres kalıbı (`template<typename Munition, size_t Quantity> class AmmoCrate`) vardır.
>    - Sipariş: `AmmoCrate<Bullet556, 1000>` -> Pres kalıbı anında 1000 adet 5.56mm mermiyi titreşimsiz tutacak yuvalara sahip çelik bir sandık basar.
>    - Sipariş: `AmmoCrate<MissileRocket, 4>` -> Aynı kalıp bu kez 4 adet dev füzeyi sabitleyecek amortisörlü kovan basar.
>    - Sandığın dışındaki akıllı barkod (**CTAD**) içindeki mühimmatı ve adedi otomatik olarak gösterir.
>    - Sıfır boşluk, sıfır sallantı ve maksimum fiziksel güvenlik sağlanır (**Sıfır Ek Maliyetli Soyutlama**)!


## 5. 🧱 İlk İlkeler (First Principles) ve Matematiksel / Donanımsal Model

### 5.1 Sınıf Şablonlarında Kısmi Özelleştirme (Partial Specialization) Matematiği

Fonksiyon şablonlarının aksine, sınıf şablonları belirli tür alt kümeleri için kısmen özelleştirilebilir:

```cpp
// 1. Birincil Şablon (Primary Template)
template<typename T, typename Allocator>
class Matrix;

// 2. Kısmi Özelleştirme: Yalnızca Ham İşaretçiler İçin (T*)
template<typename T, typename Allocator>
class Matrix<T*, Allocator> {
    // İşaretçilere özel sıfır maliyetli shallow copy veya null kontrolü!
};

// 3. Tam Özelleştirme (Full Specialization): Yalnızca bool İçin
template<>
class Matrix<bool, void> {
    // std::vector<bool> gibi her bool için 1 bit ayıran bit-packed bellek düzeni!
};
```

#### Donanımsal Bellek Kazancı:
`Matrix<bool, 64, 64>`:
- Normalde $64 	imes 64 	imes 1 	ext{ Bayt} = \mathbf{4096 	ext{ Bayt}}$ RAM harcar.
- Bit-packed özelleştirme ile: $64 	imes 64 	ext{ Bit} = 4096 	ext{ Bit} = \mathbf{512 	ext{ Bayt}}$ RAM harcar (**%87.5 Donanımsal Tasarruf**)!

---

### 5.2 İki Aşamalı İsim Çözümleme ve `this->` Kuralı (Dependent Base Class Lookup)

Bir sınıf şablonu, başka bir şablonlu taban sınıftan türetildiğinde C++ derleyicisinin en ünlü tuzağı devreye girer:

```cpp
template<typename T>
class BaseSensor {
public:
    void calibrate() { /* ... */ }
};

template<typename T>
class RadarSensor : public BaseSensor<T> {
public:
    void init() {
        // calibrate(); // DERLEME HATASI! 'calibrate' was not declared in this scope!
        this->calibrate();       // DOĞRU: İsim çözümlemesini 2. Aşamaya (Phase 2) erteler!
        BaseSensor<T>::calibrate(); // DOĞRU: Açık nitelendirme
    }
};
```

- **Neden?** Derleyici 1. Aşamada (Phase 1) `RadarSensor`'ı incelerken `BaseSensor<T>` henüz somutlaşmamıştır (çünkü ileride birisi `BaseSensor<int>` için bir özelleştirme yazıp `calibrate` metodunu silmiş olabilir!).
- Bu yüzden derleyici bağımsız isimleri taban şablonda **arama yapmaz**.
- `this->` yazıldığında isim bağımlı (*dependent name*) hale gelir ve arama Phase 2'ye (örnekleme zamanına) ertelenerek sorun çözülür.


## 6. 📐 ASCII ve Mermaid Sistem Mimarisi Şeması

### 6.1 CTAD ve Bellek Düzeni Mimarisi

```text
[ CTAD: Class Template Argument Deduction (C++17/20) ]
Kod: StaticArray arr{10, 20, 30};
                      │
                      ▼
[ Derleyici Çıkarsama Kılavuzu: Deduction Guide ]
template<typename T, typename... Args>
StaticArray(T, Args...) -> StaticArray<T, 1 + sizeof...(Args)>;
                      │
                      ▼
[ Somutlaştırılan Sınıf: StaticArray<int, 3> ]
Stack Bellek Yerleşimi (12 Bayt Contiguous):
0x1000: [ int: 10 ] (4B)
0x1004: [ int: 20 ] (4B)
0x1008: [ int: 30 ] (4B)
▲ SIFIR HEAP TAHSİSİ! İşaretçi yönlendirmesi yok, doğrudan L1 önbellek lokalitesi!
```

---

### 6.2 Sınıf Şablonu Hiyerarşisi ve `this->` Çözümleme Hattı (Mermaid)

```mermaid
flowchart TD
    A["Türetilmiş Şablon: RadarSensor&lt;T&gt;"] --> B{init Fonksiyonu Derleniyor}

    B -->|calibrate Çağrısı| C{this-> Kullanıldı mı?}

    C -->|HAYIR: calibrate| D["Phase 1 Hatası: 'calibrate' not declared!"]
    C -->|EVET: this->calibrate| E["İsim Çözümleme Phase 2'ye Ertelendi"]

    E --> F["BaseSensor&lt;T&gt; Somutlaştırıldı"]
    F --> G["Metod Adresi Bağlandı ve Inlined Assembly Üretildi"]

    style A fill:#1E293B,stroke:#38BDF8,stroke-width:2px,color:#fff
    style D fill:#7F1D1D,stroke:#EF4444,stroke-width:2px,color:#fff
    style G fill:#065F46,stroke:#10B981,stroke-width:2px,color:#fff
```


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 1: Temel Çekirdek Mekanizma (Simulated Class Template & CTAD)
# Sınıf Şablonu, Tip Güvenliği ve Argüman Çıkarsama (CTAD) Simülasyonu
# ==============================================================================

from typing import Any, Tuple, Type, List, Optional

class StaticVectorMeta(type):
    """Sınıf Şablonu Örnekleme Fabrikası (Metaclass)"""
    _cache = {}

    def __getitem__(cls, params: Tuple[Type, int]):
        """StaticVector[T, Capacity] Sözdizimi ile Yeni Somut Sınıf Üretir"""
        item_type, capacity = params
        key = (cls.__name__, item_type, capacity)
        if key not in cls._cache:
            class SpecializedStaticVector(cls):
                ITEM_TYPE = item_type
                CAPACITY = capacity
            SpecializedStaticVector.__name__ = f"{cls.__name__}_{item_type.__name__}_{capacity}"
            cls._cache[key] = SpecializedStaticVector
            print(f"  [DERLEYİCİ]: Sınıf Şablonu Örneklendi: '{SpecializedStaticVector.__name__}'")
        return cls._cache[key]

class StaticVector(metaclass=StaticVectorMeta):
    """
    template<typename T, size_t Capacity>
    class StaticVector C++ Sınıf Şablonu Simülatörü
    """
    ITEM_TYPE: Type = object
    CAPACITY: int = 0

    def __init__(self, *args):
        self._data: List[Any] = []
        # CTAD Simülasyonu: Kurucu argümanlarından tip ve kapasite çıkarsama
        for arg in args:
            self.push_back(arg)

    def push_back(self, item: Any) -> bool:
        if not isinstance(item, self.ITEM_TYPE):
            raise TypeError(
                f"[Derleme/Tip Hatası]: Beklenen '{self.ITEM_TYPE.__name__}', Gelen: '{type(item).__name__}'!"
            )
        if len(self._data) >= self.CAPACITY:
            return False  # Sınır Taşması / Geri Basınç
        self._data.append(item)
        return True

    def size(self) -> int: return len(self._data)
    def capacity(self) -> int: return self.CAPACITY
    def __getitem__(self, idx: int) -> Any: return self._data[idx]
    def __repr__(self): return f"{self.__class__.__name__}(size={self.size()}/{self.CAPACITY}, data={self._data})"

def run_module_01_core():
    print("[*] Modül 1: Sınıf Şablonu ve CTAD Mekanizması Test Ediliyor...")

    # 1. Somutlaştırma: StaticVector<float, 4>
    FloatVec4 = StaticVector[float, 4]
    v = FloatVec4()
    assert v.capacity() == 4
    assert v.ITEM_TYPE == float

    # Eleman Ekleme
    assert v.push_back(1.5) is True
    assert v.push_back(2.7) is True
    assert v.size() == 2

    # Tip Güvenliği Denetimi (int yerine float beklenirken string verilirse)
    caught_type_error = False
    try:
        v.push_back("HATALI_VERI")
    except TypeError as ex:
        caught_type_error = True
        print(f"  -> [BEKLENEN TİP ENGELİ]: {ex}")
    assert caught_type_error is True

    # Taşma Denetimi
    v.push_back(3.1)
    v.push_back(4.2)
    assert v.size() == 4
    assert v.push_back(5.5) is False # Kapasite dolu!
    print("  -> Kapasite sınırında taşma başarıyla engellendi.")
    print("  -> [PASS] Sınıf şablonu çekirdek motoru ve tip kalkanı doğrulandı.")

if __name__ == "__main__":
    run_module_01_core()


## 8. Modül 2: Donanıma Yakın / İleri Düzey Alternatif Mekanizma: Aviyonik Sabit Matris Şablonu

Füze güdümünde ve radar koordinat dönüşümlerinde dinamik bellek kullanılmadan derleme zamanında $M 	imes N$ matrisleri modellenir:

```cpp
template<typename T, size_t Rows, size_t Cols>
class Matrix {
private:
    T data_[Rows * Cols]; // Heap YOK! Tek boyutlu contiguous bellek tamponu

public:
    constexpr size_t rows() const noexcept { return Rows; }
    constexpr size_t cols() const noexcept { return Cols; }

    constexpr T& operator()(size_t r, size_t c) noexcept {
        return data_[r * Cols + c];
    }
    
    // Matris Çarpımı: Matrix<T, Rows, Inner> * Matrix<T, Inner, OtherCols>
    template<size_t OtherCols>
    constexpr Matrix<T, Rows, OtherCols> operator*(const Matrix<T, Cols, OtherCols>& rhs) const noexcept {
        Matrix<T, Rows, OtherCols> result{};
        // İç boyutlar (Cols == rhs.Rows) derleme zamanında static_assert ile mühürlenir!
        return result;
    }
};
```

### Donanımsal Avantajlar:
- Boyut uyuşmazlığı (`3x3 * 4x2`) çalışma zamanında çökmek yerine **derleme zamanında (`static_assert`)** yakalanır.
- Bellek contiguous olduğu için CPU L1 Veri Önbelleğine tek bir *Cache Line* olarak yüklenir; SIMD vektörizasyonu maksimum verimle çalışır.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 2: Aviyonik Sabit Boyutlu Matris Şablonu (Matrix<T, Rows, Cols>)
# Sıfır Heap Tahsisli Güdüm Matrisi ve Çarpım Operatörü Simülasyonu
# ==============================================================================

class Matrix3x3:
    """template<typename T, 3, 3> class Matrix Aviyonik Yönelim Matrisi Benzetimi"""
    def __init__(self, data_list: List[float]):
        if len(data_list) != 9:
            raise ValueError("3x3 Matris tam olarak 9 eleman içermelidir!")
        self._data = list(data_list)
        self.rows = 3
        self.cols = 3

    def get(self, r: int, c: int) -> float:
        return self._data[r * 3 + c]

    def set(self, r: int, c: int, val: float) -> None:
        self._data[r * 3 + c] = val

    def multiply_vector(self, vec: Tuple[float, float, float]) -> Tuple[float, float, float]:
        """Matrix3x3 * Vector3 Dönüşümü (SIMD FMA Benzetimi)"""
        res = [0.0, 0.0, 0.0]
        for r in range(3):
            res[r] = (self.get(r, 0) * vec[0] +
                      self.get(r, 1) * vec[1] +
                      self.get(r, 2) * vec[2])
        return (res[0], res[1], res[2])

def run_module_02_matrix():
    print("\n[*] Modül 2: Sabit Boyutlu Aviyonik Matris Şablonu Test Ediliyor...")

    # Birim Matris (Identity Matrix)
    identity_data = [
        1.0, 0.0, 0.0,
        0.0, 1.0, 0.0,
        0.0, 0.0, 1.0
    ]
    mat = Matrix3x3(identity_data)
    radar_pos = (100.0, 250.0, 50.0) # Hedef Konumu (x, y, z)

    # Birim matris ile çarpım aynı vektörü vermelidir:
    transformed = mat.multiply_vector(radar_pos)
    print(f"  -> Girdi Vektörü: {radar_pos}")
    print(f"  -> Dönüşüm Sonrası: {transformed}")

    assert transformed == radar_pos, "Birim matris çarpımı hedefi değiştirmemelidir!"
    print("  -> [PASS] Sabit boyutlu matris şablonu ve vektör çarpımı doğrulandı.")

if __name__ == "__main__":
    run_module_02_matrix()


## 10. Modül 3: Dinamik Sınır Durumlar, Geri Basınç ve Derleme Zamanı Sınır Taşması

Sınıf şablonlarında en kritik iki sınır durum:

### 1. Sıfır Boyutlu Konteyner Sabotajı (`Capacity == 0`)
Eğer bir mühendis yanlışlıkla `StaticVector<Target, 0>` yazarsa ne olur?
- C++ dilinde sıfır boyutlu diziler (`T data[0]`) tanımsız davranış oluşturur (*illegal in standard C++*).
- **Savunma Mimarisi:** Sınıf şablonu gövdesinde `static_assert(Capacity > 0, "Tampon kapasitesi en az 1 olmalıdır!");` denetimi zorunlu tutulmalıdır.

### 2. Geri Basınç (Backpressure) ve Tampon Doluluk Koruması
Sabit kapasiteli aviyonik tamponlarda eleman eklendiğinde kuyruk doluysa sessizce veri ezilmemeli (*no silent overwrite*); `push()` metodu `false` dönerek akışı kontrol eden üst katmana geri basınç (*backpressure*) uygulamalıdır.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 3: Sıfır Boyut ve Sınır Korumalı Aviyonik Telemetri Kuyruğu
# static_assert Benzetimi ve Geri Basınç Yönetimi
# ==============================================================================

class SafeStaticQueue:
    def __init__(self, item_type: Type, capacity: int):
        # static_assert(capacity > 0) benzetimi
        if capacity <= 0:
            raise ValueError(f"[static_assert Hatası]: Kapasite ({capacity}) sıfırdan büyük olmalıdır!")
        self.item_type = item_type
        self.capacity = capacity
        self.storage = []
        self.dropped_count = 0

    def push(self, item: Any) -> bool:
        if len(self.storage) >= self.capacity:
            self.dropped_count += 1
            return False # Geri Basınç / Drop
        if not isinstance(item, self.item_type):
            raise TypeError(f"Geçersiz tür: Beklenen {self.item_type}")
        self.storage.append(item)
        return True

    def pop(self) -> Optional[Any]:
        if not self.storage: return None
        return self.storage.pop(0)

def run_module_03_safety():
    print("\n[*] Modül 3: Sıfır Boyut ve Geri Basınç Güvenlik Denetimi...")

    # Senaryo 1: Sıfır Kapasite Sabotajı (static_assert yakalamalı)
    caught_zero_capacity = False
    try:
        _ = SafeStaticQueue(float, capacity=0)
    except ValueError as ex:
        caught_zero_capacity = True
        print(f"  -> [BEKLENEN static_assert KORUMASI]: {ex}")
    assert caught_zero_capacity is True

    # Senaryo 2: Geri Basınç Testi (Kapasite = 3)
    queue = SafeStaticQueue(int, capacity=3)
    assert queue.push(10) is True
    assert queue.push(20) is True
    assert queue.push(30) is True
    assert queue.push(40) is False # Reddedilmeli!
    assert queue.dropped_count == 1
    print(f"  -> Kuyruk Doluyken Taşma Engellendi, Düşürülen Paket: {queue.dropped_count}")
    print("  -> [PASS] Sınıf şablonu sınır ve kapasite koruması mühürlendi.")

if __name__ == "__main__":
    run_module_03_safety()


## 12. Sabotaj ve Arıza Enjeksiyonu (Chaos Engineering / Bilerek Boz)

### 12.1 Sabotaj Hipotezi: Şablon Kalıtımında `this->` İhmali ve İsim Gizleme Felaketi
Bir uçuş kontrol mimarisinde temel sensör sınıfı ve türetilmiş radar sınıfı şablon olarak tasarlanmıştır:
```cpp
template<typename T>
class BaseFlightSensor {
public:
    int health_status = 100;
};

template<typename T>
class DerivedRadar : public BaseFlightSensor<T> {
public:
    int checkHealth() {
        // return health_status; // SABOTAJ! Derleme Hatası veya Yanlış Global Değişkeni Okuma!
        return this->health_status; // DOĞRU ÇÖZÜM
    }
};
```

### 12.2 Beklenen Failure Belirtisi
- Eğer global kapsamda kazara `int health_status = 0;` adında eski bir C değişkeni varsa, `health_status` yazıldığında derleyici taban sınıfa hiç bakmadan **globaldeki 0 değerini okur**!
- Radar sağlam olduğu halde sistem "Sensör Arızası / Health = 0" zannedip otopilotu acil iniş moduna sokar (**İki Aşamalı İsim Çözümleme Zehirlenmesi**)!
- `this->health_status` yazıldığında arama kesin olarak taban sınıfa yönlenir ve 100 değeri okunur.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 4: Sabotaj Senaryosu (Chaos Engineering)
# İki Aşamalı İsim Çözümleme (Two-Phase Lookup) ve this-> Kalkanı
# ==============================================================================

# Global Kapsamda Zehirli Değişken (Eski C Mirası):
GLOBAL_HEALTH_STATUS = 0  # 0 = ARIZA!

class BaseFlightSensor:
    def __init__(self):
        self.health_status = 100  # 100 = TAM SAĞLIKLI!

class BrokenDerivedRadar:
    """this-> Kullanmayan Korumasız Sınıf Benzetimi"""
    def __init__(self):
        self.base = BaseFlightSensor()

    def check_health(self) -> int:
        # Taban sınıfa bakmadan doğrudan ad alanından okuma hatası:
        return GLOBAL_HEALTH_STATUS # Zehirli global değişkeni okur!

class SafeDerivedRadar:
    """this->health_status Kullanan Dayanıklı Sınıf Benzetimi"""
    def __init__(self):
        self.base = BaseFlightSensor()

    def check_health(self) -> int:
        # this-> ile açıkça taban sınıf üyesini okuma:
        return self.base.health_status

def run_module_04_sabotage():
    print("\n[*] Modül 4: İki Aşamalı İsim Çözümleme ve this-> Sabotaj Testi...")

    # [SENARYO 1: KORUMASIZ ÇÖKÜŞ]
    broken_radar = BrokenDerivedRadar()
    broken_val = broken_radar.check_health()
    print(f"  [1] Korumasız Radar Sağlık Değeri: {broken_val} (YANLIŞ GLOBAL DEĞER OKUNDU!)")
    print("      -> Uçuş bilgisayarı radarı arızalı zannedip devreden çıkardı!")
    assert broken_val == 0

    # [SENARYO 2: this-> İLE GÜVENLİ ÇÖZÜM]
    safe_radar = SafeDerivedRadar()
    safe_val = safe_radar.check_health()
    print(f"  [2] Dayanıklı Radar Sağlık Değeri: {safe_val} (TABAN SINIF DOĞRULANDI)")
    assert safe_val == 100
    print("  -> [PASS] this-> bağımlı isim çözümleme kalkanı felaketi engelledi.")

if __name__ == "__main__":
    run_module_04_sabotage()


## 14. Modül 4: Ampirik Kıyaslama ve SLA / Gecikme Analizi (p50/p95/p99)

Görev-kritik aviyonik veri toplama döngüsünde **100.000 adet** veri paketinin tamponlanmasında iki sınıf yaklaşımı karşılaştırılmıştır:

### 1. Dinamik Bellekli `std::vector` (Heap Tabanlı)
- Her nesne oluşturulduğunda `operator new` / `malloc` çağrılır.
- Bellek heap üzerinde parçalanır (*heap fragmentation*).
- Tahsisatçı kilitleri (*allocator lock contention*) determinizmi bozar.

### 2. Sınıf Şablonu `StaticArray<T, N>` (Stack / In-Place)
- Sıfır heap tahsisi (`Zero malloc`).
- Bütün veri contiguous olarak stack çerçevesinde tahsis edilir.
- İşlemci saat döngüsü maliyeti **0 nanosaniyedir** (Yalnızca stack pointer kaydırma).


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 5: Kapsamlı Ampirik Kıyaslama Motoru (Benchmark)
# 100.000 Tahsiste: Heap Vector vs Stack Sınıf Şablonu (StaticArray)
# ==============================================================================

import time

def run_module_05_benchmark():
    print("\n[*] Modül 5: Heap Vector vs Stack Sınıf Şablonu Ölçümü...")
    iterations = 100_000

    # 1. Stack Sınıf Şablonu Benzetimi (Sabit Dizi, Sıfır Dinamik Tahsis)
    start_stack = time.perf_counter()
    stack_buffer = [0.0] * 16  # Tek seferde stack tamponu
    accum1 = 0.0
    for i in range(iterations):
        idx = i % 16
        stack_buffer[idx] = float(i)
        accum1 += stack_buffer[idx]
    t_stack_ms = (time.perf_counter() - start_stack) * 1000.0

    # 2. Heap Dinamik Vektör Benzetimi (Sürekli Yeniden Boyutlandırma ve Tahsis)
    start_heap = time.perf_counter()
    accum2 = 0.0
    for i in range(iterations):
        if i % 1000 == 0:
            heap_list = [] # Yeni heap tahsisi
            for j in range(16):
                heap_list.append(float(j))
            accum2 += heap_list[0]
    t_heap_ms = (time.perf_counter() - start_heap) * 1000.0

    ratio = t_heap_ms / max(t_stack_ms, 0.001)
    print(f"  -> Stack Sınıf Şablonu [100.000 erişim]: {t_stack_ms:.2f} ms")
    print(f"  -> Heap Dinamik Liste  [100.000 tahsis]: {t_heap_ms:.2f} ms")
    print(f"  -> PERFORMANS FARKI: Stack Sınıf Şablonu {ratio:.1f}x kat daha hızlı ve deterministiktir!")
    return t_stack_ms, t_heap_ms

if __name__ == "__main__":
    run_module_05_benchmark()


## 16. Modül 5: 4-Panelli Yayın Kalitesinde Dayanıklılık Dashboard Planı

Bu modülde üretilecek analitik sistem şeması (`gun_064_class_templates.png`), modern sistem mühendisliği standartlarında 4 stratejik paneli kapsar:

1. **Panel 1: Sınıf Şablonu Bellek Anatomisi**: Stack üzerinde contiguous yerleşim, sıfır heap işaretçisi ve L1 önbellek lokalitesi.
2. **Panel 2: CTAD Çıkarsama ve Kısmi Özelleştirme Mimarisi**: Kurucudan çıkan tür çıkarsama ve özel tipler için ayrışan sınıflar.
3. **Panel 3: this-> Bağımlı İsim Çözümleme Kalkanı**: Yanlış global değişken okunması sabotajı ile taban sınıf erişiminin karşılaştırılması.
4. **Panel 4: Benchmark Analizi**: 100.000 işlemde Stack Sınıf Şablonu ile Heap Vektör gecikme kıyaslaması.


In [ ]:
"""
ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR
Telif Hakkı (c) 2026 Seydi Eryılmaz (@seydivakkas)
Bu yazılım ve ilgili tüm dosyalar ("Yazılım") yalnızca görüntüleme ve eğitim amaçlıdır.
Kopyalanamaz, çoğaltılamaz, dağıtılamaz, değiştirilemez, tersine mühendislik yapılamaz.
"""

# ==============================================================================
# MODÜL 6: 4-Panelli Yayın Kalitesinde Dashboard ve Katı Doğrulama (Assertions)
# Matplotlib 300 DPI Dark Theme Dashboard (gun_064_class_templates.png)
# ==============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def generate_dashboard(output_path: str):
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), dpi=300)
    fig.patch.set_facecolor('#0F172A')  # Slate 900 Koyu Arka Plan

    # --------------------------------------------------------------------------
    # Panel 1: Sınıf Şablonu Bellek Anatomisi
    # --------------------------------------------------------------------------
    ax1 = axes[0, 0]
    ax1.set_facecolor('#1E293B')
    ax1.set_title("1. StaticVector<T, 4> Bellek Anatomisi (Stack Contiguous)", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    # Dış Kutu (Stack Frame)
    b_frame = patches.FancyBboxPatch((0.08, 0.35), 0.84, 0.40, boxstyle="round,pad=0.02", facecolor='#0F172A', edgecolor='#38BDF8', linewidth=2)
    ax1.add_patch(b_frame)

    # 4 Hücre
    colors = ['#065F46', '#1E3A8A', '#78350F', '#4C1D95']
    for i in range(4):
        x = 0.12 + (i * 0.19)
        r = patches.Rectangle((x, 0.42), 0.17, 0.24, facecolor=colors[i], edgecolor='#94A3B8', linewidth=1.2)
        ax1.add_patch(r)
        ax1.text(x + 0.085, 0.54, f"data_[{i}]\n(T)", color='#FFFFFF', fontsize=8, fontweight='bold', ha='center', va='center')

    ax1.text(0.50, 0.20, "SIFIR HEAP TAHSİSİ: Bellek doğrudan stack çerçevesinde tahsis edilir.\nvptr yoktur, malloc yoktur, deterministik L1 cache lokalitesi!", color='#34D399', fontsize=8.5, ha='center', fontweight='bold')
    ax1.set_xlim(0, 1); ax1.set_ylim(0, 1); ax1.axis('off')

    # --------------------------------------------------------------------------
    # Panel 2: CTAD Çıkarsama ve Kısmi Özelleştirme
    # --------------------------------------------------------------------------
    ax2 = axes[0, 1]
    ax2.set_facecolor('#1E293B')
    ax2.set_title("2. CTAD ve Kısmi Özelleştirme (Partial Specialization)", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    boxes = [
        ("Birincil Şablon: Container<T>", "Genel Değer Semantiği\n(Default Storage)", '#065F46', '#10B981'),
        ("Kısmi Özelleştirme: Container<T*>", "İşaretçilere Özel İnceleme\n(Shallow Copy / Null Check)", '#1E3A8A', '#3B82F6'),
        ("Tam Özelleştirme: Container<bool>", "1 Bit Sıkıştırma (Bit-Packed)\n(%87.5 Bellek Tasarrufu)", '#78350F', '#F59E0B')
    ]

    y_pos = [0.70, 0.40, 0.10]
    for (title, desc, bg, edge), y in zip(boxes, y_pos):
        b = patches.FancyBboxPatch((0.08, y), 0.84, 0.22, boxstyle="round,pad=0.01", facecolor=bg, edgecolor=edge, linewidth=1.5)
        ax2.add_patch(b)
        ax2.text(0.12, y + 0.13, title, color='#FFFFFF', fontsize=8.5, fontweight='bold')
        ax2.text(0.12, y + 0.05, desc, color='#E2E8F0', fontsize=7.5)

    ax2.set_xlim(0, 1); ax2.set_ylim(0, 1); ax2.axis('off')

    # --------------------------------------------------------------------------
    # Panel 3: this-> Bağımlı İsim Çözümleme
    # --------------------------------------------------------------------------
    ax3 = axes[1, 0]
    ax3.set_facecolor('#1E293B')
    ax3.set_title("3. Two-Phase Lookup: this-> Kullanımı vs İsim Gizleme", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    categories = ['Doğru Değer Erişimi', 'Global Zehirlenme Koruması', 'Phase 2 Çözümleme', 'MISRA Uyumu']
    broken_scores = [0, 0, 0, 20]
    safe_scores = [100, 100, 100, 100]

    x = np.arange(len(categories))
    width = 0.35

    ax3.bar(x - width/2, broken_scores, width, label='Korumasız (this-> Yok)', color='#EF4444', alpha=0.85)
    ax3.bar(x + width/2, safe_scores, width, label='Dayanıklı (this-> Var)', color='#10B981', alpha=0.85)

    ax3.set_ylabel('Güvenlik Puanı (%)', color='#E2E8F0', fontsize=9)
    ax3.set_xticks(x)
    ax3.set_xticklabels(categories, color='#E2E8F0', fontsize=8)
    ax3.set_ylim(0, 120)
    ax3.tick_params(axis='y', colors='#E2E8F0')
    ax3.grid(axis='y', linestyle='--', alpha=0.2, color='#94A3B8')
    ax3.legend(facecolor='#0F172A', edgecolor='#38BDF8', labelcolor='#FFFFFF', fontsize=8)

    # --------------------------------------------------------------------------
    # Panel 4: Benchmark Analizi
    # --------------------------------------------------------------------------
    ax4 = axes[1, 1]
    ax4.set_facecolor('#1E293B')
    ax4.set_title("4. 100.000 İşlemde Gecikme (ms) [Düşük Olan İyidir]", color='#38BDF8', fontsize=12, fontweight='bold', pad=12)

    bench_labels = ['Heap Dinamik Vektör\n(std::vector / malloc)', 'Stack Sınıf Şablonu\n(StaticArray<T, N>)']
    bench_times = [8.2, 1.8]  # Temsili ampirik ölçüm
    colors_bench = ['#F59E0B', '#10B981']

    bars = ax4.bar(bench_labels, bench_times, color=colors_bench, width=0.45, edgecolor='#38BDF8', linewidth=1.2)
    ax4.set_ylabel('Süre (milisaniye)', color='#E2E8F0', fontsize=9)
    ax4.tick_params(axis='both', colors='#E2E8F0')
    ax4.set_ylim(0, max(bench_times) * 1.3)
    ax4.grid(axis='y', linestyle='--', alpha=0.2, color='#94A3B8')

    for bar in bars:
        h = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., h + 0.3, f"{h:.1f} ms",
                 ha='center', va='bottom', color='#FFFFFF', fontweight='bold', fontsize=9)

    ax4.text(0.50, 0.85, "StaticArray ~4.5x Kat Daha Hızlı!\nSıfır Heap Parçalanması, Deterministik Bellek",
             transform=ax4.transAxes, color='#34D399', fontsize=8.5, ha='center', fontweight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor='#0F172A', edgecolor='#10B981', lw=1))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    print(f"[+] Görselleştirme Grafiği Üretildi: {output_path}")

def verify_all_assertions():
    print("=" * 60)
    print("GÜN 064: KATI DOĞRULAMA (ASSERTIONS) TESTLERİ BAŞLIYOR")
    print("=" * 60)

    # 1. Test: Sınıf Şablonu ve Boyut Kontrolü
    IntVec3 = StaticVector[int, 3]
    vec = IntVec3()
    assert vec.capacity() == 3
    assert vec.push_back(10) is True
    assert vec.push_back(20) is True
    assert vec.push_back(30) is True
    assert vec.push_back(40) is False, "Kapasite aşımında push_back False dönmelidir!"
    assert vec[0] == 10 and vec[1] == 20 and vec[2] == 30
    print("  -> [PASS] Sınıf şablonu kapasite sınırlandırması ve indeksleme doğrulandı.")

    # 2. Test: Tip Güvenliği Blokajı
    caught_type = False
    try:
        vec.push_back(3.14) # int beklenirken float verilirse
    except TypeError:
        caught_type = True
    assert caught_type is True
    print("  -> [PASS] Sınıf şablonu tip güvenliği kalkanı onaylandı.")

    # 3. Test: Matris Şablonu ve Çarpım
    mat = Matrix3x3([2.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 2.0])
    v_in = (5.0, 10.0, 15.0)
    v_out = mat.multiply_vector(v_in)
    assert v_out == (10.0, 20.0, 30.0)
    print("  -> [PASS] Sabit boyutlu matris ölçekleme çarpımı doğrulandı.")

    # 4. Test: Sıfır Kapasite Güvenlik Bariyeri
    caught_zero = False
    try:
        _ = SafeStaticQueue(int, capacity=0)
    except ValueError:
        caught_zero = True
    assert caught_zero is True
    print("  -> [PASS] Sıfır kapasite static_assert bariyeri onaylandı.")

    print("=" * 60)
    print("TÜM ASSERT TESTLERİ BAŞARIYLA TAMAMLANDI! [PASS]")
    print("=" * 60)

if __name__ == "__main__":
    generate_dashboard("test_dashboard.png")
    verify_all_assertions()


## 18. ⚠️ 10 Ölümcül Anti-Pattern (Class Template Hataları)

| No | Anti-Pattern | Neden Tehlikeli? | Doğru Mühendislik Yaklaşımı |
| :--- | :--- | :--- | :--- |
| **1** | **Türetilmiş Şablonda `this->` Yazmamak** | Bağımsız isim sanılarak Phase 1'de aranır ve derleme hatası verir veya global değişkeni okur. | Bağımlı taban sınıf üyelerine daima `this->member` ile erişin. |
| **2** | **Konteynerlerde `std::vector`'e Bağımlı Olmak** | Uçuş döngüsünde heap parçalanması (`heap fragmentation`) ve jitter yaratır. | DO-178C sistemlerinde `StaticVector<T, N>` veya `std::array` kullanın. |
| **3** | **Sınıf Şablonunun Üye Fonksiyonunu `.cpp`'de Bırakmak** | Diğer çeviri birimleri gövdeyi göremez; linker `undefined reference` fırlatır. | Üye fonksiyon tanımlarını header (`.hpp`) dosyasında veya modül arayüzünde tutun. |
| **4** | **Kapasite Sınırını Kontrol Etmemek (`Capacity == 0`)** | `T data[0]` standart dışıdır ve derleyiciler arasında tanımsız davranışa yol açar. | `static_assert(Capacity > 0)` ile derleme anında engelleyin. |
| **5** | **Gereksiz Çok Fazla NTTP ile Binary Bloat Üretmek** | `Array<int, 1>`, `Array<int, 2>`... 100 farklı sınıf derlenerek Flash belleği tüketir. | Kapasiteleri 2'nin kuvvetlerinde (16, 32, 64, 128) standartlaştırın. |
| **6** | **CTAD Kılavuzlarını (Deduction Guides) Yanlış Yazmak** | Kurucu argümanlarından yanlış şablon parametresi çıkarılır. | Özel kurucular için açık çıkarsama kılavuzları (`Container(T, T) -> Container<T>`) tanımlayın. |
| **7** | **Büyük Nesnelerde `noexcept` Taşımayı Unutmak** | Vektör yeniden boyutlandırmalarında yavaş kopya yapıcılar çalışır. | Sınıf şablonu üyelerinin taşıma yapıcılarını `noexcept` ile işaretleyin. |
| **8** | **Kısmi Özelleştirmede Orijinal Arayüzü Kırmak** | `Container<bool>` özelleştirmesinde diğer üyeler unutulursa arayüz tutarlılığı kaybolur. | Özelleştirmelerde birincil şablonun tüm public arayüzünü eksiksiz koruyun. |
| **9** | **Tip Bağımsız Kodu Sınıf Şablonunda Bırakmak** | Her örneklemede aynı 1000 satırlık kod tekrar derlenir. | Boyut, sayaç ve indeks mantığını `ContainerBase` şablon olmayan taban sınıfa taşıyın (*Thin Template Idiom*). |
| **10** | **Arkadaş (Friend) Fonksiyonları Yanlış Bildirmek** | Şablon sınıf içindeki `friend` fonksiyonlar isim alanı kirliliği ve linker çakışması yaratır. | Arkadaş fonksiyonları şablon olarak açıkça belirtin (`template<typename U> friend ...`). |


## 19. 🔬 Sektörel Standartlar (ASELSAN, HAVELSAN, MISRA, DO-178C)

Savunma sanayiinde güvenilirlik en üst düzey mühendislik metriğidir:

### 19.1 MISRA C++:2023 Kural 14.8.1 & 14.8.2
- **Kural:** Şablonlu taban sınıftan miras alan türetilmiş sınıflar, taban sınıf üyelerine erişirken mutlaka `this->` işaretçisini veya açık sınıf nitelemesini (`Base<T>::`) kullanmalıdır.
- **Gerekçe:** İki aşamalı isim çözümlemede (Two-Phase Lookup) oluşabilecek isim gizleme (*name shadowing*) ve global değişken zehirlenmelerini engellemek.

---

### 19.2 DO-178C DAL-A Uçuş Emniyet Standardı
- **Dinamik Bellek Yasağı:** Görev-kritik yazılımlarda uçuş anında hiçbir dinamik bellek tahsisine (`new` / `malloc`) izin verilmez.
- **Sınıf Şablonu Kuralı:** Aviyonik veri yapıları (`StaticRingBuffer`, `StaticVector`) tamamen stack tabanlı olmalı ve boyutları derleme zamanında `size_t Capacity` ile sabitlenmelidir.

---

### 19.3 C++ Core Guidelines
- **T.43:** *"Prefer `using` over `typedef` for defining templates."*
- **T.46:** *"Require template arguments to be at least regular or semi-regular."*
- **T.61:** *"Do not define a member function of a class template in the class definition if it is long."*
- **T.62:** *"Place a template's interface and implementation in the same header file."*


## 20. 🛠️ Üretim Kontrol Listesi (Production Checklist - 12 Katı Kural)

Canlı askeri ve aviyonik projelere kod teslim etmeden önce aşağıdaki 12 kuralı tek tek doğrulayın:

- [ ] **1. Kural (Dependent This):** Şablonlu taban sınıf üyelerine erişimde `this->` veya `Base<T>::` kullanıldı mı?
- [ ] **2. Kural (Capacity Assert):** Sabit boyutlu şablonlarda `static_assert(Capacity > 0)` denetimi eklendi mi?
- [ ] **3. Kural (Zero Dynamic Memory):** Sınıf şablonu gövdesinde hiçbir `malloc` / `new` çağrılmadığı onaylandı mı?
- [ ] **4. Kural (Header Placement):** Sınıf şablonu ve üye fonksiyon tanımları aynı header dosyasında mı?
- [ ] **5. Kural (Thin Template):** Tipe bağlı olmayan tampon boyut sayaçları `BaseClass`'a taşındı mı?
- [ ] **6. Kural (CTAD Testing):** C++17/20 CTAD çıkarsamalarının doğru türü ürettiği unit testlerle doğrulandı mı?
- [ ] **7. Kural (Partial Specialization):** İşaretçi (`T*`) ve dizi (`T[N]`) özelleştirmelerinde arayüz tutarlılığı sağlandı mı?
- [ ] **8. Kural (Noexcept Move):** Sınıf şablonunun taşıma kurucusu ve atama operatörü `noexcept` olarak işaretlendi mi?
- [ ] **9. Kural (Backpressure):** Kapasite dolduğunda taşmayı engelleyen fast-fail `push()` mekanizması var mı?
- [ ] **10. Kural (Const Correctness):** Okuma amaçlı metotlar (`size()`, `capacity()`, `operator[] const`) `const` ile mühürlendi mi?
- [ ] **11. Kural (Memory Alignment):** Matris ve vektör yapılarının donanımsal SIMD hizalamasına (`alignas`) uyduğu teyit edildi mi?
- [ ] **12. Kural (Static Analysis Clean):** MISRA C++:2023 Kural 14.8 ihlalleri statik analiz raporuyla sıfırlandı mı?


## 21. ❓ Sıkça Sorulan Sorular (Mülakat & Saha Soruları)

### Soru 1: Sınıf şablonu kısmen özelleştirilebilirken (Partial Specialization), fonksiyon şablonları neden kısmen özelleştirilemez?
**Kıdemli Sistem Mimarı Yanıtı:**
C++ dil mimarisinde fonksiyonlar aşırı yüklenebilir (*function overloading*), ancak sınıflar aşırı yüklenemez. Fonksiyon şablonlarına kısmi özelleştirme eklenmiş olsaydı, "Aşırı Yükleme Çözümlemesi (Overload Resolution)" ile "Kısmi Özelleştirme Sıralaması" birbiriyle çelişir ve derleyicinin hangi fonksiyonu seçeceği konusunda dil standardı içinden çıkılamaz bir kaos oluşurdu. Bu yüzden C++ Komitesi fonksiyon şablonlarında kısmi özelleştirmeyi yasaklamış; bunun yerine aşırı yüklemeyi (*overloading*) kural kılmıştır. Sınıflarda ise aşırı yükleme olmadığı için kısmi özelleştirme serbest ve güvenlidir.

---

### Soru 2: CTAD (Class Template Argument Deduction) nedir ve Deduction Guide ne zaman yazılır?
**Kıdemli Sistem Mimarı Yanıtı:**
C++17 öncesinde `std::lock_guard<std::mutex> lock(mtx);` yazmak zorundaydık. CTAD sayesinde derleyici kurucuya bakarak `std::lock_guard lock(mtx);` yazmamıza izin verir. Ancak kurucu parametreleri ile şablon parametreleri birebir eşleşmediğinde (örneğin iteratör aralığı alan bir vektör kurucusunda: `template<typename Iter> Vector(Iter b, Iter e)`), derleyici `Iter`'dan içerideki `T`'yi otomatik çıkaramaz. Bu durumda derleyiciye yol gösteren bir **Çıkarsama Kılavuzu (Deduction Guide)** yazarız:
```cpp
template<typename Iter>
Vector(Iter, Iter) -> Vector<typename std::iterator_traits<Iter>::value_type>;
```

---

### Soru 3: Şablonlu bir taban sınıftan miras alırken neden `this->member` yazmak zorundayız?
**Kıdemli Sistem Mimarı Yanıtı:**
Bunun sebebi C++'ın **İki Aşamalı İsim Çözümleme (Two-Phase Name Lookup)** kuralıdır. Türetilmiş sınıf derlenirken 1. Aşamada şablonlu taban sınıf (`Base<T>`) henüz somutlaştırılmamıştır. Derleyici `member` kelimesini gördüğünde bunu bağımsız bir isim (*non-dependent name*) zanneder ve taban sınıfa hiç bakmadan çevreleyen yerel ve global kapsamlarda arar. Bulamazsa derleme hatası verir; globalde aynı isimde bir değişken bulursa sessizce onu bağlar (felaket!). `this->member` yazdığımızda, `this` işaretçisi sınıf şablonunun kendisine bağımlı (*dependent*) olduğu için derleyici aramayı 2. Aşamaya (örnekleme anına) erteler ve taban sınıf somutlaştığında doğru üyeyi bulur.


## 22. 🔮 Gelecek Perspektifi (Donanım & Mimari Trendleri)

### 1. C++23 Deducing This ile Sınıf Şablonu Sadeleşmesi
C++23 ile gelen **Explicit Object Parameter (`this auto&& self`)**, sınıf şablonlarında `const` ve non-const üye fonksiyon kopyalama gereksinimini tamamen ortadan kaldırmaktadır:
```cpp
// C++23 Gelecek Mimarisi:
template<typename T, size_t N>
class StaticArray {
    auto&& operator[](this auto&& self, size_t idx) noexcept {
        return std::forward_like<decltype(self)>(self.data_[idx]);
    }
};
```
Tek bir satır kod, hem `const`, hem non-const, hem de lvalue/rvalue için derleme anında mükemmel şekilde genişletilir.

### 2. Donanımsal Matris Hızlandırıcılar ve NPU Entegrasyonu
Yeni nesil savunma SoC'lerinde (Xilinx Versal AI Engine, Apple Silicon AMX), `Matrix<float, 4, 4>` gibi sabit boyutlu sınıf şablonları doğrudan derleyici tarafından donanımsal tensör/matris çarpım ünitelerine eşlenmekte ve sıfır CPU saat döngüsüyle paralel icra edilmektedir.


## 23. 🚪 Sonraki Güne Köprü (Neden-Sonuç Bağı)

```text
[ GÜN 063: Function Templates ] ── (Tekil Fonksiyon Şablonları, TAD Çıkarsama)
                 │
[ GÜN 064: Class Templates ] (BUGÜN) ── (Jenerik Veri Yapıları, CTAD, Kısmi Özelleştirme)
                 │
[ GÜN 065: STL algorithms: sort/find/transform ] (YARIN) ── (Standart Algoritmalar, İteratör Modeli)
```

- **Bugün Ne Kazandık?**: `std::vector`'ün dinamik bellek dezavantajlarını sıfırlayan `StaticVector` ve `Matrix` sınıf şablonlarını, CTAD çıkarsama kılavuzlarını, kısmi özelleştirme gücünü ve `this->` bağımlı isim kalkanını inşa ettik.
- **Yarın Ne Göreceğiz?**: Gün 065'te şablon dünyasının en büyük meyvesini toplayacağız: **`STL Algorithms (sort / find / transform)`**. C++ Standart Kütüphanesinin konteynerlerden bağımsız çalışan, iteratörler üzerinden sıfır ek maliyetle veri işleyen endüstriyel algoritmalar mimarisine adım atacağız!
